# steering-resistance — Colab launcher

Thin launcher: it clones the repo, installs the `steer` package, and runs the tested pipeline. **No science lives in this notebook** — all logic is in the package, so a Colab run and a local run are identical.

**Before running:** set a GPU runtime (Runtime → Change runtime type → GPU), and add two Colab secrets (🔑 in the left sidebar): `HF_TOKEN` and `WANDB_API_KEY` (both optional — leave tracking off if you skip them).

In [ ]:
!nvidia-smi -L    # confirm a GPU is attached

In [ ]:
# Clone (or update) the repo and install the package. Colab already ships a
# CUDA torch, so we don't reinstall it — pip just adds/upgrades the rest.
import os, subprocess
REPO = "https://github.com/JacoDuToit11/steering-resistance.git"
if not os.path.isdir("steering-resistance"):
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
%cd steering-resistance
!git pull -q
!pip install -q -e ".[dev,track]"

In [ ]:
# Auth (optional). Pulls tokens from Colab secrets into the env; skip a token
# and that tracker just stays off (the pipeline is fail-soft).
from google.colab import userdata
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: set")
    except Exception:
        print(f"{key}: not found (that tracker will stay off)")

In [ ]:
# Machinery check (fast) — the three gradient-flow asserts on the 0.5B smoke config.
!python scripts/verify_hooks.py configs/smoke_0.5b.yaml

In [ ]:
# Point tracking/backup at your accounts, then run. Set HF_USER to enable the
# adapter push; leave it None to keep the run local. Edit CONFIG / WANDB freely.
import yaml
CONFIG = "configs/qwen7b_popqa.yaml"
HF_USER = None            # e.g. "JacoDuToit11" → pushes to <user>/steer-qwen7b-popqa
WANDB_PROJECT = "steering-resistance"   # or None to disable W&B

cfg = yaml.safe_load(open(CONFIG))
run = os.path.splitext(os.path.basename(CONFIG))[0]
cfg["hub_repo_id"] = f"{HF_USER}/steer-{run}" if HF_USER else None
cfg["wandb_project"] = WANDB_PROJECT
yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)
print("hub_repo_id:", cfg["hub_repo_id"], "| wandb_project:", cfg["wandb_project"])

!python scripts/run.py {CONFIG}